# Chapter 9: Screening calls

This notebook runs every piece of code in the chapter, from the saved results, in order. Read the chapter alongside it: [Screening calls](https://books.sugiv.fyi/voice_applications/screening-calls.html).


In [ ]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the notebook
# from a clone of it. Either way it moves into the book's folder and installs what the chapter needs.
import os
import pathlib
import sys

if "google.colab" in sys.modules and not pathlib.Path("/content/voices-of-the-future").exists():
    !git clone -q --depth 1 https://github.com/sugix/voices-of-the-future /content/voices-of-the-future
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/voices-of-the-future")]
root = next(p for p in candidates if (p / "voice" / "sim").is_dir())
os.chdir(root / "voice")
%pip install -q -r ../requirements/results.txt

## Who rings a bike shop

In [ ]:
import json
import re
import sys
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, "sim")
from scoring import normalize  # Chapter 4

heard = json.load(open("data/ch9/heard.json"))  # what Parakeet heard in each call's first 20 s
judged = json.load(open("data/ch9/judged.json"))  # Qwen3-4B's answers
ids = list(heard)
unwanted = np.array([not heard[i]["wanted"] for i in ids])
print(Counter((heard[i]["source"], "unwanted" if not heard[i]["wanted"] else "wanted") for i in ids), "\n")
for i in ["r1098320", "l01", "l03", "l05"]:
    print(f"{i:9s} {heard[i]['category']:15s} {heard[i]['heard'][:120]}")

## A list of words

In [ ]:
SCAM_WORDS = ["press 1", "press one", "press 2", "press two", "warranty", "social security", "amazon", "irs",
              "lawsuit", "arrest", "suspend", "final notice", "credit card", "interest rate", "medicare", "loan",
              "vehicle", "refund", "account", "debt"]
by_words = np.array([any(w in heard[i]["heard"].lower() for w in SCAM_WORDS) for i in ids])
print(f"robocalls caught: {np.sum(by_words & unwanted)} of {unwanted.sum()}")
print(f"genuine callers blocked: {np.sum(by_words & ~unwanted)} of {(~unwanted).sum()}")
for i in ids:
    if by_words[ids.index(i)] and heard[i]["wanted"]:
        hit = [w for w in SCAM_WORDS if w in heard[i]["heard"].lower()]
        print(f"   {heard[i]['category']:15s} {hit}  {heard[i]['heard'][:80]}")

## Asking the model

In [ ]:
print(judged["system"], "\n\nQuestion:", judged["verdict"])

In [ ]:
def auc(score, positive):
    """Chance that a random unwanted call scores higher than a random wanted one."""
    pos, neg = score[positive], score[~positive]
    return float(np.mean([(p > n) + 0.5 * (p == n) for p in pos for n in neg]))


verdict = np.array([judged["calls"][i]["verdict"] for i in ids])
print(f"AUC {auc(verdict, unwanted):.3f}")
for t in [0.5, 0.9]:
    print(f"block when P(yes) >= {t}: robocalls caught {np.sum((verdict >= t) & unwanted)} of {unwanted.sum()}, "
          f"genuine callers blocked {np.sum((verdict >= t) & ~unwanted)} of {(~unwanted).sum()}")
print("\nrobocalls the model was surest were fine:")
for i in sorted([i for i in ids if not heard[i]["wanted"]], key=lambda i: judged["calls"][i]["verdict"])[:6]:
    print(f"   P(yes)={judged['calls'][i]['verdict']:.3f}  {heard[i]['heard'][:110]}")
print("\ngenuine callers it was surest were spam:")
for i in sorted([i for i in ids if heard[i]["wanted"]], key=lambda i: -judged["calls"][i]["verdict"])[:2]:
    print(f"   P(yes)={judged['calls'][i]['verdict']:.3f}  {heard[i]['category']}: {heard[i]['heard'][:90]}")

## Asking better questions

In [ ]:
QUESTIONS = judged["questions"]
X = np.array([[judged["calls"][i]["answers"][q] for q in QUESTIONS] for i in ids])
print(f"{'question':12s} {'robocalls':>10s} {'genuine':>8s}   (average P(yes))")
for k, q in enumerate(QUESTIONS):
    print(f"{q:12s} {X[unwanted, k].mean():10.2f} {X[~unwanted, k].mean():8.2f}   {QUESTIONS[q][:70]}")

In [ ]:
def fit(X, y, l2=0.1, steps=3000, lr=0.5):
    """Logistic regression by gradient descent, on log-odds features."""
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(steps):
        p = 1 / (1 + np.exp(-(X @ w + b)))
        w -= lr * (X.T @ (p - y) / len(y) + l2 * w)
        b -= lr * np.mean(p - y)
    return w, b


logodds = np.log(np.clip(X, 1e-4, 1 - 1e-4) / np.clip(1 - X, 1e-4, 1))
folds = np.random.default_rng(0).permutation(len(ids)) % 5
score = np.zeros(len(ids))
for f in range(5):
    w, b = fit(logodds[folds != f], unwanted[folds != f].astype(float))
    score[folds == f] = 1 / (1 + np.exp(-(logodds[folds == f] @ w + b)))
w, b = fit(logodds, unwanted.astype(float))  # weights on the whole bench, to read them
print(f"AUC on unseen calls {auc(score, unwanted):.3f}\n")
for q, wq in sorted(zip(QUESTIONS, w), key=lambda t: -abs(t[1])):
    print(f"{q:12s} {wq:+.2f}")
for t in [0.5, 0.9]:
    print(f"\nblock when score >= {t}: robocalls caught {np.sum((score >= t) & unwanted)} of {unwanted.sum()}, "
          f"genuine callers blocked {np.sum((score >= t) & ~unwanted)} of {(~unwanted).sum()}", end="")

## Robocalls repeat themselves

In [ ]:
def shingles(text):
    words = re.sub(r"\d", "#", normalize(text)).split()
    return {tuple(words[k:k + 3]) for k in range(len(words) - 2)}


def jaccard(a, b):
    return len(a & b) / max(1, len(a | b))


order = list(np.random.default_rng(1).permutation(ids))  # every call, in some order of arrival
known_scams, repeats = [], {}
for i in order:
    s = shingles(heard[i]["heard"])
    repeats[i] = any(jaccard(s, t) >= 0.6 for t in known_scams)
    if not heard[i]["wanted"]:
        known_scams.append(s)  # a person labels it a scam once they've heard it
robo = [i for i in ids if heard[i]["source"] == "NCSU robocall"]
print(f"robocalls matching a script already labelled as a scam: {sum(repeats[i] for i in robo)} of {len(robo)}")
print(f"genuine callers matching one: {sum(repeats[i] for i in ids if heard[i]['wanted'])}")

## What never to block

In [ ]:
k = ids.index("l03")
print(heard["l03"]["heard"], "\n")
print({q: round(float(X[k, j]), 2) for j, q in enumerate(QUESTIONS)})

In [ ]:
A = {q: X[:, j] for j, q in enumerate(QUESTIONS)}


def decide(k):
    i = ids[k]
    if A["relevant"][k] >= 0.5 or A["specific"][k] >= 0.5:
        return "answer"  # about the shop or its business: never block
    if repeats[i]:
        return "block"  # a script already labelled as a scam
    if A["institution"][k] >= 0.5:
        return "take a message"  # a person decides, with an impersonation warning
    if score[k] >= 0.9:
        return "block"
    return "take a message" if score[k] >= 0.5 else "answer"


decision = np.array([decide(k) for k in range(len(ids))])
for side, mask in [("robocalls", unwanted), ("genuine callers", ~unwanted)]:
    print(f"{side:16s}", {d: int(np.sum(decision[mask] == d)) for d in ["block", "take a message", "answer"]})
print("\ngenuine callers that didn't get a normal answer:")
for k, i in enumerate(ids):
    if heard[i]["wanted"] and decision[k] != "answer":
        print(f"   {decision[k]:15s} {heard[i]['category']:15s} {heard[i]['heard'][:80]}")

## Rerun the models

Everything above used the saved results. The cells below rerun the models that produced them, on a GPU (in Colab:
Runtime, then Change runtime type, then T4 GPU). Each step builds the software it needs in its own environment the
first time, which takes a few minutes, then runs. The first cell below sets up a two-item check that everything
works, on a scratch copy of the folder, so the saved results stay as they are. Delete its lines for a full rerun,
which overwrites the saved results with yours. On GPUs without bfloat16, such as the T4, the scripts use float16
automatically.

In [ ]:
# A two-item check that every step runs, on a scratch copy so the saved results stay untouched.
# Delete these lines for a full rerun, which overwrites the saved results with your own.
import os, shutil, tempfile
os.environ["VOTF_QUICK"] = "2"
scratch = os.path.join(tempfile.gettempdir(), "votf-quick")
shutil.copytree(".", scratch, dirs_exist_ok=True)
os.chdir(scratch)

Downloads 150 robocalls (public domain) from the NCSU dataset.

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/capture_ch9.py --part fetch

In [ ]:
!bash tools/stack.sh tts
!~/.votf/envs/tts/bin/python sim/capture_ch9.py --part voice

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/capture_ch9.py --part hear

In [ ]:
!bash tools/stack.sh hf
!~/.votf/envs/hf/bin/python sim/capture_ch9.py --part judge